In [1]:
!pip install "mcp<2.0.0" fastmcp langchain-mcp-adapters langchain-google-genai langgraph

INFO: pip is looking at multiple versions of fastmcp-slim[client,server] to determine which version is compatible with other requirements. This could take a while.
  Using cached fastmcp-4.0.10-py3-none-any.whl.metadata (9.0 kB)
INFO: pip is still looking at multiple versions of fastmcp-slim[client,server] to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.6/234.6 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 769.4/769.4 kB 31.7 MB/s eta 0:00:00
  Attempting uninstall: mcp
    Found existing installation: mcp 2.2.0
    Uninstalling mcp-2.2.0:
      Successfully uninstalled mcp-2.2.0
  Attempting uninstall: fastmcp-slim
    Found existing installation: fastmcp-slim 4.0.10
    Uninstalling fastmcp-slim-4.0.10:
      Successfully uninstalled fastmcp-slim-4.0.10
  Attempting uninstall: fastmcp
    Found existing installation: fastmcp 4.0.10
    Uninstalling fastmcp-4.0.10:
      S

In [2]:
%%writefile academic_server.py
from fastmcp import FastMCP

# Initialize an MCP server instance named "AcademicAdvisorServer"
mcp = FastMCP("AcademicAdvisorServer")

@mcp.tool()
def calculate_grade(assignments_avg: float, midterm_score: float, final_exam: float) -> str:
    """Calculates final weighted grade: Assignments 30%, Midterm 30%, Final Exam 40%."""
    final_score = (assignments_avg * 0.30) + (midterm_score * 0.30) + (final_exam * 0.40)

    if final_score >= 85:
        letter_grade = "A"
    elif final_score >= 70:
        letter_grade = "B"
    elif final_score >= 60:
        letter_grade = "C"
    else:
        letter_grade = "F"

    return f"Numerical Average: {final_score:.2f}, Letter Grade: {letter_grade}"

@mcp.tool()
def check_prerequisites(course_code: str) -> str:
    """Returns official prerequisite requirements for Computer Science courses."""
    prerequisites = {
        "CS101": "None",
        "CS201": "CS101",
        "CS301": "CS201",
        "CS401": "CS301",
        "MATH201": "MATH101"
    }

    course = course_code.upper().strip()
    reqs = prerequisites.get(course, "Course not found in the catalog.")
    return f"Prerequisites for {course}: {reqs}"

if __name__ == "__main__":
    # Run the server using the STDIO transport
    mcp.run()

Overwriting academic_server.py


In [6]:
import os
import sys
import asyncio
from google.colab import userdata
from langchain_mcp_adapters.client import MultiServerMCPClient

# 1. Patch Colab's standard streams to prevent the 'fileno' subprocess crash
sys.stdout.fileno = lambda: 1
sys.stderr.fileno = lambda: 2
sys.stdin.fileno = lambda: 0

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

async def test_discovery():
    server_config = {
        "academic_tools": {
            "transport": "stdio",
            "command": "python",
            "args": ["academic_server.py"]
        }
    }

    # 2. Instantiate the client directly
    client = MultiServerMCPClient(server_config)

    # 3. Dynamically fetch MCP tools
    tools = await client.get_tools()

    # 4. Verify tools were discovered
    print(f"Discovered {len(tools)} tools:")
    for t in tools:
        print(f"- Name: {t.name} | Description: {t.description}")

await test_discovery()

Discovered 2 tools:
- Name: calculate_grade | Description: Calculates final weighted grade: Assignments 30%, Midterm 30%, Final Exam 40%.
- Name: check_prerequisites | Description: Returns official prerequisite requirements for Computer Science courses.


In [8]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent

async def run_advisor_agent(user_query: str):
    server_config = {
        "academic_tools": {
            "transport": "stdio",
            "command": "python",
            "args": ["academic_server.py"]
        }
    }

    # 1. Instantiate the client directly
    client = MultiServerMCPClient(server_config)

    # 2. Discover tools
    tools = await client.get_tools()

    # 3. Initialize ChatGoogleGenerativeAI
    model = ChatGoogleGenerativeAI(model="gemini-3.8-flash")

    # 4. Create a ReAct agent using create_react_agent
    agent = create_react_agent(model, tools)

    # 5. Invoke the agent with the user query
    result = await agent.ainvoke({"messages": [("user", user_query)]})
    return result

# Test Query
query = (
    "I want to take CS301 next term. What prerequisites do I need? "
    "Also, my assignment average was 88, midterm was 74, and final exam was 90. What is my final grade?"
)

response = await run_advisor_agent(query)
print("\n--- Final Agent Response ---")
print(response["messages"][-1].content)

/tmp/ipykernel_2952/2491545573.py:23: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(model, tools)



--- Final Agent Response ---
[{'type': 'text', 'text': 'Here are the details for your requests:\n\n### **CS301 Prerequisites**\n* **Prerequisite Course:** **CS201**\n\n---\n\n### **Final Grade Calculation**\n* **Assignment Average (30%):** 88\n* **Midterm Exam (30%):** 74\n* **Final Exam (40%):** 90\n\n* **Numerical Average:** **84.60**\n* **Letter Grade:** **B**', 'extras': {'signature': 'Eq4BCqsBAWkUfRNI6HtqmGSr20w7qZaFf/he9IC71Br98VyRKKJyKeMCiP7/MOufFMo/cR/MkT7QnQ29SX9tp42O00VPtXDfqId4XfLCAHiFKP3wMHrHnJzC5605bseA1GY4bGb70ST14iImwY83p7xVKaB7pHjKAhJRm8GL+hdSyJN3j+1OA/jYzxUWdyIHzt5uV8m9d6FFvXamavVkc2xKdORvvGSNXh6WS0lBYUek'}}]


In [9]:
# Inspect intermediate message payloads from the Phase 3 run
print("=== Phase 4: Intermediate Message Inspection ===")
for msg in response["messages"]:
    # Inspect LLM tool invocation arguments
    if msg.type == "ai" and msg.tool_calls:
        print("\n[AIMessage -> Tool Request]")
        for call in msg.tool_calls:
            print(f"Tool Name: {call['name']}")
            print(f"Arguments: {call['args']}")
    # Inspect MCP server return payloads
    elif msg.type == "tool":
        print(f"\n[ToolMessage -> MCP Response ({msg.name})]")
        print(f"Content: {msg.content}")

=== Phase 4: Intermediate Message Inspection ===

[AIMessage -> Tool Request]
Tool Name: check_prerequisites
Arguments: {'course_code': 'CS301'}
Tool Name: calculate_grade
Arguments: {'midterm_score': 74, 'assignments_avg': 88, 'final_exam': 90}

[ToolMessage -> MCP Response (check_prerequisites)]
Content: [{'type': 'text', 'text': 'Prerequisites for CS301: CS201', 'id': 'lc_579e83a9-f969-44d2-8498-91f247d0fea5'}]

[ToolMessage -> MCP Response (calculate_grade)]
Content: [{'type': 'text', 'text': 'Numerical Average: 84.60, Letter Grade: B', 'id': 'lc_c57b1644-06f6-45e2-afb5-046a150b9d42'}]


In [10]:
%%writefile academic_server.py
from fastmcp import FastMCP

mcp = FastMCP("AcademicAdvisorServer")

@mcp.tool()
def calculate_grade(assignments_avg: float, midterm_score: float, final_exam: float) -> str:
    """Calculates final weighted grade: Assignments 30%, Midterm 30%, Final Exam 40%."""
    final_score = (assignments_avg * 0.30) + (midterm_score * 0.30) + (final_exam * 0.40)

    if final_score >= 85:
        letter_grade = "A"
    elif final_score >= 70:
        letter_grade = "B"
    elif final_score >= 60:
        letter_grade = "C"
    else:
        letter_grade = "F"

    return f"Numerical Average: {final_score:.2f}, Letter Grade: {letter_grade}"

@mcp.tool()
def check_prerequisites(course_code: str) -> str:
    """Returns official prerequisite requirements for Computer Science courses."""
    prerequisites = {
        "CS101": "None",
        "CS201": "CS101",
        "CS301": "CS201",
        "CS401": "CS301",
        "MATH201": "MATH101"
    }
    course = course_code.upper().strip()
    reqs = prerequisites.get(course, "Course not found in the catalog.")
    return f"Prerequisites for {course}: {reqs}"

# Tool 3: Algebraic score targeting
@mcp.tool()
def required_exam_score(assignments_avg: float, midterm_score: float, target_grade: float) -> str:
    """Calculates the minimum final exam score needed to achieve a target overall grade."""
    required = (target_grade - (0.3 * assignments_avg) - (0.3 * midterm_score)) / 0.4
    if required > 100:
        return f"You need {required:.1f}% on the final exam, which is mathematically impossible."
    elif required <= 0:
        return "You have already achieved this target grade even with a 0 on the final exam."
    return f"You need to score at least {required:.1f}% on the final exam."

if __name__ == "__main__":
    mcp.run()

Overwriting academic_server.py


In [11]:
# Verify dynamic discovery of the new tool without touching LangGraph logic
scaffold_query = (
    "I have an assignment average of 82 and a midterm score of 75. "
    "What score do I need on the final exam to get an 85 overall in the class?"
)

scaffold_response = await run_advisor_agent(scaffold_query)

print("\n--- Agent Answer ---")
print(scaffold_response["messages"][-1].content)

print("\n--- Tool Execution Trace ---")
for msg in scaffold_response["messages"]:
    if msg.type == "ai" and msg.tool_calls:
        print(f"Called: {msg.tool_calls[0]['name']} with {msg.tool_calls[0]['args']}")
    elif msg.type == "tool":
        print(f"Output: {msg.content}")

/tmp/ipykernel_2952/2491545573.py:23: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(model, tools)



--- Agent Answer ---
[{'type': 'text', 'text': 'To achieve an overall grade of **85%** in the class, you will need to score at least **94.8%** on the final exam.', 'extras': {'signature': 'EmkKZwFpFH0Tor4Yd7DnjG9Jp91G3GzvY1L3JH4u/A1ohghjmCSjyTTQXlyi3uld10i1yNuPMbjVYB4u3FQGdj1YXs7a/aZ5grYjCo1SjCfJ3U/l0nd+yh85hRIAkuas6m895qCY0CZmHtA='}}]

--- Tool Execution Trace ---
Called: required_exam_score with {'assignments_avg': 82, 'target_grade': 85, 'midterm_score': 75}
Output: [{'type': 'text', 'text': 'You need to score at least 94.8% on the final exam.', 'id': 'lc_9508b635-a64a-4bb9-afcd-b6d117956889'}]
